# SLM Medical Domain - Stage A on Google Colab

This notebook runs the verified Stage A trainer. Use a GPU runtime. The dataset is extracted to Colab's local SSD; completed checkpoints are mirrored to Google Drive.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("BF16 supported:", torch.cuda.is_bf16_supported())

PyTorch: 2.11.0+cu128
CUDA runtime: 12.8
GPU: Tesla T4
VRAM GB: 14.6
BF16 supported: True


## Mount Drive and configure paths

Upload `stage-a-data.tar` to `MyDrive/medical-slm/` before running this cell.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

REPOSITORY_URL = "https://github.com/moshiur00/small-language-model-for-medical-domain"
PROJECT_DIRECTORY = Path("/content/medical-slm")
DATA_ARCHIVE = Path("/content/drive/MyDrive/medical-slm/stage-a-data.tar")
DRIVE_CHECKPOINTS = Path("/content/drive/MyDrive/medical-slm-runs/stage_a_dev/checkpoints")

assert DATA_ARCHIVE.is_file(), f"Missing data archive: {DATA_ARCHIVE}"

Mounted at /content/drive


## Clone, install, and extract local training data

For a private repository, configure Git credentials through a Colab secret. Never place an access token directly in the notebook.

In [ ]:
import os
import shutil
import subprocess
import sys

if not PROJECT_DIRECTORY.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_DIRECTORY)], check=True)

os.chdir(PROJECT_DIRECTORY)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
source_directory = str(PROJECT_DIRECTORY / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)
subprocess.run(["tar", "-xf", str(DATA_ARCHIVE), "-C", str(PROJECT_DIRECTORY)], check=True)
print("Working directory:", Path.cwd())

Working directory: /content/medical-slm


In [ ]:
required = [
    Path("artifacts/tokenizer/tokenizer.json"),
    Path("datasets/tokenized/stage_a/dataset_manifest.json"),
    Path("datasets/tokenized/stage_a/train/metadata.json"),
    Path("datasets/tokenized/evaluation/dataset_manifest.json"),
    Path("datasets/tokenized/evaluation/validation/metadata.json"),
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f"Missing required files: {missing}"
shards = list(Path("datasets/tokenized/stage_a/train").glob("shard_*.bin"))
assert len(shards) == 115, f"Expected 115 Stage A shards, found {len(shards)}"
print("Dataset verified: 115 Stage A shards")

Dataset verified: 115 Stage A shards


## Run the focused cloud-training tests

In [ ]:
subprocess.run([
    "python", "-m", "pytest",
    "tests/test_training_checkpoint.py",
    "tests/test_training_step.py",
    "tests/test_stage_a_trainer.py",
    "-q",
], check=True)

CompletedProcess(args=['python', '-m', 'pytest', 'tests/test_training_checkpoint.py', 'tests/test_training_step.py', 'tests/test_stage_a_trainer.py', '-q'], returncode=0)

## Create the isolated 100-update development profile

In [ ]:
import yaml

development = yaml.safe_load(Path("configs/training_stage_a_colab.yaml").read_text())
development.update({
    "output_directory": "/content/stage_a_dev",
    "checkpoint_backup_directory": str(DRIVE_CHECKPOINTS),
    "max_updates": 100,
    "validation_interval": 50,
    "checkpoint_interval": 50,
})
DEVELOPMENT_CONFIG = Path("/content/training_stage_a_colab_dev.yaml")
DEVELOPMENT_CONFIG.write_text(yaml.safe_dump(development, sort_keys=False))
print(DEVELOPMENT_CONFIG.read_text())

train_directory: datasets/tokenized/stage_a/train
validation_directory: datasets/tokenized/evaluation/validation
tokenizer_json: artifacts/tokenizer/tokenizer.json
output_directory: /content/stage_a_dev
checkpoint_backup_directory: /content/drive/MyDrive/medical-slm-runs/stage_a_dev/checkpoints
seed: 42
device: auto
precision: auto
micro_batch_size: 16
gradient_accumulation_steps: 8
evaluation_batch_size: 32
dataloader_workers: 2
pin_memory: true
learning_rate: 0.0003
final_learning_rate: 3.0e-05
warmup_updates: 73
total_updates: 7310
max_updates: 100
max_epochs: 1
adam_beta1: 0.9
adam_beta2: 0.95
weight_decay: 0.1
max_gradient_norm: 1.0
log_interval: 10
validation_interval: 50
checkpoint_interval: 50
keep_recent_checkpoints: 2
milestone_interval: 1000



## Initial validation baseline

Run this once before development training. Random-initialization loss should be near `ln(16000) ≈ 9.68`.

In [ ]:
import math
import yaml
from medical_slm.model import DecoderConfig
from medical_slm.training.config import load_stage_a_config
from medical_slm.training.trainer import StageATrainer

training_config = load_stage_a_config(DEVELOPMENT_CONFIG)
model_values = yaml.safe_load(Path("configs/model_stage_a.yaml").read_text())
model_config = DecoderConfig.from_mapping(model_values)
baseline_trainer = StageATrainer(training_config, model_config)
baseline = baseline_trainer.evaluate()
baseline_trainer.metric_logger.close()
expected_random_loss = math.log(model_config.vocab_size)
expected_validation_tokens = 1_822 * 256
print({"loss": baseline.loss, "expected_random_loss": expected_random_loss,
       "perplexity": baseline.perplexity, "tokens": baseline.tokens})
assert math.isfinite(baseline.loss), "Initial validation loss is not finite."
assert math.isfinite(baseline.perplexity), "Initial perplexity is not finite."
assert baseline.tokens == expected_validation_tokens, (
    f"Expected {expected_validation_tokens} validation tokens, got {baseline.tokens}."
)
assert abs(baseline.loss - expected_random_loss) < 1.0, (
    f"Initial loss {baseline.loss:.4f} is unexpectedly far from "
    f"ln(vocab_size)={expected_random_loss:.4f}."
)
print("Initial validation baseline gate: PASSED")

{'loss': 9.774141221616977, 'expected_random_loss': 9.680344001221918, 'perplexity': 17573.392047848276, 'tokens': 466432}
Initial validation baseline gate: PASSED


## Development run: train to update 50

The Colab profile mirrors the verified update-50 checkpoint to Drive automatically.

In [ ]:
subprocess.run([
    "python", "scripts/training/train_stage_a.py",
    "--config", str(DEVELOPMENT_CONFIG),
    "--max-updates", "50",
], check=True)

CompletedProcess(args=['python', 'scripts/training/train_stage_a.py', '--config', '/content/training_stage_a_colab_dev.yaml', '--max-updates', '50'], returncode=0)

In [ ]:
assert (DRIVE_CHECKPOINTS / "latest.json").is_file()
print("Drive checkpoint backup:", DRIVE_CHECKPOINTS)
print((DRIVE_CHECKPOINTS / "latest.json").read_text())

Drive checkpoint backup: /content/drive/MyDrive/medical-slm-runs/stage_a_dev/checkpoints
{
  "checkpoint": "checkpoint_00000050"
}



In [ ]:
import json
from pathlib import Path

records = [
    json.loads(line)
    for line in Path("/content/stage_a_dev/metrics.jsonl").read_text().splitlines()
    if line.strip()
]

for record in records:
    if record["event"] == "validation":
        metrics = record["metrics"]
        print(
            f"update={record['update']}",
            f"loss={metrics['loss']:.4f}",
            f"perplexity={metrics['perplexity']:.2f}",
            f"best={metrics['is_best']}",
        )

update=0 loss=9.7741 perplexity=17573.39 best=True
update=50 loss=7.4758 perplexity=1764.80 best=True
update=100 loss=6.8605 perplexity=953.80 best=True


In [ ]:
from medical_slm.training.precision import resolve_precision

policy = resolve_precision("auto", "cuda")
print(policy.name, policy.uses_grad_scaler)

fp16 True


In [ ]:
from pathlib import Path

local_full = Path("/content/stage_a/checkpoints")
drive_full = Path(
    "/content/drive/MyDrive/medical-slm-runs/stage_a/checkpoints"
)

print("Local full checkpoint exists:", (local_full / "latest.json").exists())
print("Drive full checkpoint exists:", (drive_full / "latest.json").exists())

Local full checkpoint exists: False
Drive full checkpoint exists: False


## Restart/resume test

Restart the Colab session, then rerun the setup cells above. Run this cell to restore the mirrored checkpoint into local storage.

In [ ]:
LOCAL_CHECKPOINTS = Path("/content/stage_a_dev/checkpoints")
LOCAL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "rsync", "-a", f"{DRIVE_CHECKPOINTS}/", f"{LOCAL_CHECKPOINTS}/"
], check=True)
assert (LOCAL_CHECKPOINTS / "latest.json").is_file()
print("Restored:", (LOCAL_CHECKPOINTS / "latest.json").read_text())

Restored: {
  "checkpoint": "checkpoint_00000050"
}



In [ ]:
subprocess.run([
    "python", "scripts/training/train_stage_a.py",
    "--config", str(DEVELOPMENT_CONFIG),
    "--resume", "latest",
    "--max-updates", "100",
], check=True)

CompletedProcess(args=['python', 'scripts/training/train_stage_a.py', '--config', '/content/training_stage_a_colab_dev.yaml', '--resume', 'latest', '--max-updates', '100'], returncode=0)

# Resume full run - completely standalone

In [ ]:
from google.colab import drive

import json
import os
import subprocess
import sys
from pathlib import Path

import torch
import yaml

# ---------- Configuration ----------

REPOSITORY_URL = (
    "https://github.com/moshiur00/"
    "small-language-model-for-medical-domain.git"
)

PROJECT_DIRECTORY = Path("/content/medical-slm")

DATA_ARCHIVE = Path(
    "/content/drive/MyDrive/medical-slm/stage-a-data.tar"
)

LOCAL_OUTPUT = Path("/content/stage_a")

LOCAL_CHECKPOINTS = (
    LOCAL_OUTPUT / "checkpoints"
)

DRIVE_CHECKPOINTS = Path(
    "/content/drive/MyDrive/"
    "medical-slm-runs/stage_a/checkpoints"
)

# ---------- Mount Drive ----------

drive.mount("/content/drive")

assert DATA_ARCHIVE.is_file(), (
    f"Dataset archive is missing: {DATA_ARCHIVE}"
)

assert (
    DRIVE_CHECKPOINTS / "latest.json"
).is_file(), (
    "No full-run checkpoint exists in Drive."
)

# ---------- Clone or update GitHub ----------

if not (PROJECT_DIRECTORY / ".git").is_dir():
    subprocess.run([
        "git",
        "clone",
        REPOSITORY_URL,
        str(PROJECT_DIRECTORY),
    ], check=True)
else:
    subprocess.run([
        "git",
        "-C",
        str(PROJECT_DIRECTORY),
        "pull",
        "--ff-only",
    ], check=True)

os.chdir(PROJECT_DIRECTORY)

# ---------- Install ----------

subprocess.run([
    "python",
    "-m",
    "pip",
    "install",
    "-q",
    "-e",
    ".[dev]",
], check=True)

source_directory = str(PROJECT_DIRECTORY / "src")

if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

# ---------- Extract dataset locally ----------

subprocess.run([
    "tar",
    "-xf",
    str(DATA_ARCHIVE),
    "-C",
    str(PROJECT_DIRECTORY),
], check=True)

# ---------- Verify inputs ----------

required_inputs = [
    Path("configs/training_stage_a_colab.yaml"),
    Path("configs/model_stage_a.yaml"),
    Path("artifacts/tokenizer/tokenizer.json"),
    Path("datasets/tokenized/stage_a/dataset_manifest.json"),
    Path("datasets/tokenized/stage_a/train/metadata.json"),
    Path("datasets/tokenized/evaluation/dataset_manifest.json"),
    Path(
        "datasets/tokenized/evaluation/"
        "validation/metadata.json"
    ),
]

missing = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

assert not missing, f"Missing required inputs: {missing}"

assert torch.cuda.is_available(), (
    "Select a GPU runtime before training."
)

# ---------- Recreate identical FP16 config ----------

config = yaml.safe_load(
    Path(
        "configs/training_stage_a_colab.yaml"
    ).read_text()
)

config.update({
    "precision": "fp16",
    "output_directory": str(LOCAL_OUTPUT),
    "checkpoint_backup_directory": str(
        DRIVE_CHECKPOINTS
    ),
    "max_updates": 7310,
    "validation_interval": 250,
    "checkpoint_interval": 250,
})

FULL_CONFIG = Path(
    "/content/training_stage_a_colab_full.yaml"
)

FULL_CONFIG.write_text(
    yaml.safe_dump(config, sort_keys=False)
)

# ---------- Restore checkpoint locally ----------

LOCAL_CHECKPOINTS.mkdir(
    parents=True,
    exist_ok=True,
)

subprocess.run([
    "rsync",
    "-a",
    f"{DRIVE_CHECKPOINTS}/",
    f"{LOCAL_CHECKPOINTS}/",
], check=True)

latest_pointer = json.loads(
    (
        LOCAL_CHECKPOINTS / "latest.json"
    ).read_text()
)

latest_name = latest_pointer["checkpoint"]

from medical_slm.training.checkpoint import (
    verify_checkpoint,
)

verify_checkpoint(
    LOCAL_CHECKPOINTS / latest_name
)

state = json.loads(
    (
        LOCAL_CHECKPOINTS
        / latest_name
        / "trainer_state.json"
    ).read_text()
)

print("GPU:", torch.cuda.get_device_name(0))
print("Configured precision: fp16")
print("Restored checkpoint:", latest_name)
print("Resume update:", state["update"])
print("Consumed tokens:", state["consumed_tokens"])

# ---------- Resume ----------

subprocess.run([
    "python",
    "scripts/training/train_stage_a.py",
    "--config",
    str(FULL_CONFIG),
    "--resume",
    "latest",
], check=True)


# Fresh full run  completely standalone




In [ ]:
from google.colab import drive

import os
import subprocess
import sys
from pathlib import Path

import torch
import yaml

# ---------- Configuration ----------

REPOSITORY_URL = (
    "https://github.com/moshiur00/"
    "small-language-model-for-medical-domain.git"
)

PROJECT_DIRECTORY = Path("/content/medical-slm")

DATA_ARCHIVE = Path(
    "/content/drive/MyDrive/medical-slm/stage-a-data.tar"
)

LOCAL_OUTPUT = Path("/content/stage_a")

DRIVE_CHECKPOINTS = Path(
    "/content/drive/MyDrive/"
    "medical-slm-runs/stage_a/checkpoints"
)

# ---------- Mount Drive ----------

drive.mount("/content/drive")

assert DATA_ARCHIVE.is_file(), (
    f"Dataset archive is missing: {DATA_ARCHIVE}"
)

# ---------- Clone or update GitHub ----------

if not (PROJECT_DIRECTORY / ".git").is_dir():
    subprocess.run([
        "git",
        "clone",
        REPOSITORY_URL,
        str(PROJECT_DIRECTORY),
    ], check=True)
else:
    subprocess.run([
        "git",
        "-C",
        str(PROJECT_DIRECTORY),
        "pull",
        "--ff-only",
    ], check=True)

os.chdir(PROJECT_DIRECTORY)

# ---------- Install ----------

subprocess.run([
    "python",
    "-m",
    "pip",
    "install",
    "-q",
    "-e",
    ".[dev]",
], check=True)

source_directory = str(PROJECT_DIRECTORY / "src")

if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

# ---------- Extract dataset locally ----------

subprocess.run([
    "tar",
    "-xf",
    str(DATA_ARCHIVE),
    "-C",
    str(PROJECT_DIRECTORY),
], check=True)

# ---------- Verify required inputs ----------

required_inputs = [
    Path("configs/training_stage_a_colab.yaml"),
    Path("configs/model_stage_a.yaml"),
    Path("artifacts/tokenizer/tokenizer.json"),
    Path("datasets/tokenized/stage_a/dataset_manifest.json"),
    Path("datasets/tokenized/stage_a/train/metadata.json"),
    Path("datasets/tokenized/evaluation/dataset_manifest.json"),
    Path(
        "datasets/tokenized/evaluation/"
        "validation/metadata.json"
    ),
]

missing = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

assert not missing, f"Missing required inputs: {missing}"

shards = list(
    Path("datasets/tokenized/stage_a/train").glob(
        "shard_*.bin"
    )
)

assert len(shards) == 115, (
    f"Expected 115 shards, found {len(shards)}"
)

assert torch.cuda.is_available(), (
    "Select a GPU runtime before training."
)

# ---------- Create portable full-run config ----------

config = yaml.safe_load(
    Path(
        "configs/training_stage_a_colab.yaml"
    ).read_text()
)

config.update({
    # FP16 works on T4, A100 and L4 and remains
    # compatible if Colab changes GPU after reconnecting.
    "precision": "fp16",
    "output_directory": str(LOCAL_OUTPUT),
    "checkpoint_backup_directory": str(
        DRIVE_CHECKPOINTS
    ),
    "max_updates": 7310,
    "validation_interval": 250,
    "checkpoint_interval": 250,
})

FULL_CONFIG = Path(
    "/content/training_stage_a_colab_full.yaml"
)

FULL_CONFIG.write_text(
    yaml.safe_dump(config, sort_keys=False)
)

# ---------- Verify fresh-run state ----------

print("GPU:", torch.cuda.get_device_name(0))
print(
    "Capability:",
    torch.cuda.get_device_capability()
)
print("Configured precision: fp16")
print("Stage A shards:", len(shards))

assert not (
    LOCAL_OUTPUT / "checkpoints/latest.json"
).exists(), (
    "A local full-run checkpoint exists. "
    "Use the resume cell."
)

assert not (
    DRIVE_CHECKPOINTS / "latest.json"
).exists(), (
    "A Drive full-run checkpoint exists. "
    "Use the resume cell."
)

# ---------- Start fresh training ----------

subprocess.run([
    "python",
    "scripts/training/train_stage_a.py",
    "--config",
    str(FULL_CONFIG),
], check=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Capability: (7, 5)
Configured precision: fp16
Stage A shards: 115


CompletedProcess(args=['python', 'scripts/training/train_stage_a.py', '--config', '/content/training_stage_a_colab_full.yaml'], returncode=0)

# Verify After Training

## Restore The Environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = (
    "https://github.com/moshiur00/"
    "small-language-model-for-medical-domain.git"
)

PROJECT_DIRECTORY = Path("/content/medical-slm")

DATA_ARCHIVE = Path(
    "/content/drive/MyDrive/medical-slm/stage-a-data.tar"
)

if not (PROJECT_DIRECTORY / ".git").is_dir():
    subprocess.run([
        "git",
        "clone",
        REPOSITORY_URL,
        str(PROJECT_DIRECTORY),
    ], check=True)
else:
    subprocess.run([
        "git",
        "-C",
        str(PROJECT_DIRECTORY),
        "pull",
        "--ff-only",
    ], check=True)

os.chdir(PROJECT_DIRECTORY)

subprocess.run([
    "python",
    "-m",
    "pip",
    "install",
    "-q",
    "-e",
    ".[dev]",
], check=True)

subprocess.run([
    "tar",
    "-xf",
    str(DATA_ARCHIVE),
    "-C",
    str(PROJECT_DIRECTORY),
], check=True)

source_directory = str(PROJECT_DIRECTORY / "src")

if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

print("Environment restored.")

Environment restored.


## Verify the Drive pointers and final state

In [ ]:
import json
from pathlib import Path

checkpoint_root = Path(
    "/content/drive/MyDrive/"
    "medical-slm-runs/stage_a/checkpoints"
)

pointers = {}

for name in [
    "latest",
    "final_stage_a",
    "best_validation",
]:
    pointers[name] = json.loads(
        (
            checkpoint_root / f"{name}.json"
        ).read_text()
    )["checkpoint"]

print(json.dumps(pointers, indent=2))

final_checkpoint = (
    checkpoint_root
    / pointers["final_stage_a"]
)

state = json.loads(
    (
        final_checkpoint
        / "trainer_state.json"
    ).read_text()
)

print(json.dumps(state, indent=2))

assert pointers["latest"] == "checkpoint_00007310"
assert pointers["final_stage_a"] == "checkpoint_00007310"
assert state["update"] == 7310
assert state["epoch"] == 1
assert state["batch_cursor"] == 0
assert state["consumed_samples"] == 935_642
assert state["consumed_tokens"] == 239_524_352
assert state["skipped_updates"] == 0
assert state["non_finite_events"] == 0

print("Final Stage A state: VERIFIED")

{
  "latest": "checkpoint_00007310",
  "final_stage_a": "checkpoint_00007310",
  "best_validation": "checkpoint_00007250"
}
{
  "batch_cursor": 0,
  "best_validation_loss": 3.19838276440017,
  "consumed_micro_batches": 58478,
  "consumed_samples": 935642,
  "consumed_tokens": 239524352,
  "epoch": 1,
  "non_finite_events": 0,
  "skipped_updates": 0,
  "update": 7310
}
Final Stage A state: VERIFIED


## Verify checkpoint integrity

In [ ]:
from medical_slm.training.checkpoint import (
    verify_checkpoint,
)

for checkpoint_name in set(pointers.values()):
    path = checkpoint_root / checkpoint_name
    manifest = verify_checkpoint(path)

    print(
        checkpoint_name,
        "verified artifacts:",
        len(manifest["artifacts"]),
    )

checkpoint_00007310 verified artifacts: 9
checkpoint_00007250 verified artifacts: 9


# Evaluation

In [ ]:
import gc
import json
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import torch
import yaml
from torch.utils.data import DataLoader

from medical_slm.data.tokenization.dataset import (
    PackedTokenDataset,
)
from medical_slm.model import DecoderConfig, DecoderModel
from medical_slm.training.evaluation import (
    evaluate_shifted_packed,
)
from medical_slm.training.precision import (
    resolve_precision,
)

assert torch.cuda.is_available()

device = torch.device("cuda")
precision = resolve_precision("auto", device)

print("GPU:", torch.cuda.get_device_name(0))
print("Evaluation precision:", precision.name)

checkpoint_root = Path(
    "/content/drive/MyDrive/"
    "medical-slm-runs/stage_a/checkpoints"
)

pointers = {
    name: json.loads(
        (
            checkpoint_root / f"{name}.json"
        ).read_text()
    )["checkpoint"]
    for name in [
        "best_validation",
        "final_stage_a",
    ]
}

print("Pointers:", pointers)

model_values = yaml.safe_load(
    Path("configs/model_stage_a.yaml").read_text()
)
model_config = DecoderConfig.from_mapping(
    model_values
)

validation_dataset = PackedTokenDataset(
    "datasets/tokenized/evaluation/validation"
)
test_dataset = PackedTokenDataset(
    "datasets/tokenized/evaluation/test"
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)


def evaluate_checkpoint(
    checkpoint_name,
    batches,
):
    model = DecoderModel(model_config)

    state_dict = torch.load(
        checkpoint_root
        / checkpoint_name
        / "model.pt",
        map_location="cpu",
        weights_only=True,
    )

    model.load_state_dict(state_dict)
    model.to(device)

    result = evaluate_shifted_packed(
        model=model,
        batches=batches,
        device=device,
        precision=precision,
    )

    del state_dict
    del model
    gc.collect()
    torch.cuda.empty_cache()

    return result

GPU: Tesla T4
Evaluation precision: fp16
Pointers: {'best_validation': 'checkpoint_00007250', 'final_stage_a': 'checkpoint_00007310'}


##Compare best and final on validation

In [ ]:
validation_results = {}

for checkpoint_name in sorted(set(pointers.values())):
    print(
        "Evaluating validation:",
        checkpoint_name,
    )

    result = evaluate_checkpoint(
        checkpoint_name,
        validation_loader,
    )

    validation_results[checkpoint_name] = result

    print(
        f"loss={result.loss:.6f}",
        f"perplexity={result.perplexity:.3f}",
        f"tokens={result.tokens}",
    )

    assert result.tokens == 466_432
    assert result.loss == result.loss

Evaluating validation: checkpoint_00007250
loss=3.198383 perplexity=24.493 tokens=466432
Evaluating validation: checkpoint_00007310
loss=3.198705 perplexity=24.501 tokens=466432


## Select the lower-loss checkpoint:

In [ ]:
selected_checkpoint = min(
    validation_results,
    key=lambda name: validation_results[name].loss,
)

selected_validation = validation_results[
    selected_checkpoint
]

print("Selected checkpoint:", selected_checkpoint)
print(
    "Selected validation loss:",
    selected_validation.loss,
)
print(
    "Selected validation perplexity:",
    selected_validation.perplexity,
)

Selected checkpoint: checkpoint_00007250
Selected validation loss: 3.19838276440017
Selected validation perplexity: 24.492887380448188


## Evaluate the selected checkpoint once on test

In [ ]:
print("Evaluating test:", selected_checkpoint)

test_result = evaluate_checkpoint(
    selected_checkpoint,
    test_loader,
)

assert test_result.tokens == 303_360
assert test_result.loss == test_result.loss

print("Test loss:", test_result.loss)
print("Test perplexity:", test_result.perplexity)
print("Test tokens:", test_result.tokens)

Evaluating test: checkpoint_00007250
Test loss: 3.679167894799388
Test perplexity: 39.61341782363614
Test tokens: 303360


## Save the report and promoted pointer
The checkpoint referenced by promoted_stage_a.json,
stage_a_evaluation.json,
The tokenizer,
Model and training configurations,
Dataset manifests,
All checkpoint pointer files.

In [ ]:
report = {
    "generated_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "selected_checkpoint": selected_checkpoint,
    "validation": {
        name: asdict(result)
        for name, result in validation_results.items()
    },
    "test": asdict(test_result),
    "source_pointers": pointers,
    "gpu": torch.cuda.get_device_name(0),
    "precision": precision.name,
}

report_path = Path(
    "/content/drive/MyDrive/"
    "medical-slm-runs/stage_a/"
    "stage_a_evaluation.json"
)

report_path.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )
    + "\n"
)

promoted_pointer = (
    checkpoint_root / "promoted_stage_a.json"
)

promoted_pointer.write_text(
    json.dumps(
        {"checkpoint": selected_checkpoint},
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

print("Saved report:", report_path)
print("Promoted:", promoted_pointer.read_text())

Saved report: /content/drive/MyDrive/medical-slm-runs/stage_a/stage_a_evaluation.json
Promoted: {
  "checkpoint": "checkpoint_00007250"
}



# Full Stage A training - Don't need it now

Run only after the 50→100 resume test succeeds and its validation loss decreases. On a fresh runtime, restore Drive checkpoints first. Omit `--resume` only when starting from random initialization.

In [ ]:
FULL_CONFIG = "configs/training_stage_a_colab.yaml"
FULL_DRIVE_CHECKPOINTS = Path("/content/drive/MyDrive/medical-slm-runs/stage_a/checkpoints")
FULL_LOCAL_CHECKPOINTS = Path("/content/stage_a/checkpoints")

# Confirm auto precision on the currently assigned GPU before starting.
from medical_slm.training.precision import resolve_precision
print("Resolved precision:", resolve_precision("auto", "cuda").name)

# Fresh full run (uncomment once):
# subprocess.run(["python", "scripts/training/train_stage_a.py",
#                 "--config", FULL_CONFIG], check=True)

# Resume after a new Colab runtime: restore the full-run checkpoint first.
# FULL_LOCAL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
# subprocess.run(["rsync", "-a", f"{FULL_DRIVE_CHECKPOINTS}/",
#                 f"{FULL_LOCAL_CHECKPOINTS}/"], check=True)
# subprocess.run(["python", "scripts/training/train_stage_a.py",
#                 "--config", FULL_CONFIG, "--resume", "latest"],
#                check=True)